# 데이터세트 결합: 연결 및 추가

데이터에 대한 가장 흥미로운 연구 중 일부는 다양한 데이터 소스를 결합하는 것에서 비롯됩니다.
이러한 작업에는 서로 다른 두 데이터세트의 매우 간단한 연결부터 데이터세트 간의 중복을 올바르게 처리하는 보다 복잡한 데이터베이스 스타일 조인 및 병합에 이르기까지 모든 작업이 포함될 수 있습니다.
'Series'와 'DataFrame'은 이러한 유형의 작업을 염두에 두고 구축되었으며 Pandas에는 이러한 종류의 데이터 랭글링을 빠르고 간단하게 만드는 기능과 메서드가 포함되어 있습니다.

여기서는 `pd.concat` 함수를 사용하여 `Series`와 `DataFrame`을 간단히 연결하는 방법을 살펴보겠습니다. 나중에 Pandas에서 구현된 보다 정교한 메모리 제 병합 및 조인에 대해 살펴보겠습니다.

표준 가져오기부터 시작합니다.

In [1]:
import pandas as pd
import numpy as np

편의를 위해 다음 예에서 유용할 특정 형식의 `DataFrame`을 생성하는 이 함수를 정의하겠습니다.

In [2]:
def make_df(cols, ind):
    """Quickly make a DataFrame"""
    data = {c: [str(c) + str(i) for i in ind]
            for c in cols}
    return pd.DataFrame(data, ind)

# example DataFrame
make_df('ABC', range(3))

,A,B,C
0,A0,B0,C0
1,A1,B1,C1
2,A2,B2,C2


또한 여러 ``DataFrame``을 나란히 표시할 수 있는 빠른 클래스를 만들겠습니다. 코드는 IPython/Jupyter가 풍부한 개체 표시를 구현하는 데 사용하는 특수 `_repr_html_` 메서드를 사용합니다.

In [3]:
class display(object):
    """Display HTML representation of multiple objects"""
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""
    def __init__(self, *args):
        self.args = args
        
    def _repr_html_(self):
        return '\n'.join(self.template.format(a, eval(a)._repr_html_())
                         for a in self.args)
    
    def __repr__(self):
        return '\n\n'.join(a + '\n' + repr(eval(a))
                           for a in self.args)
    

다음 섹션에서 논의를 계속하면 이것의 사용이 더 명확해질 것입니다.

## 회상: NumPy 배열 연결

'Series'와 'DataFrame' 개체의 연결은 NumPy 배열의 연결과 유사하게 동작하며, 이는 [NumPy 배열의 기본 사항](02.02-The-Basics-Of-NumPy-Arrays.ipynb)에서 설명한 대로 'np.concatenate' 함수를 통해 수행할 수 있습니다.
이를 사용하면 두 개 이상의 배열 내용을 단일 배열로 결합할 수 있습니다.

In [4]:
x = [1, 2, 3]
y = [4, 5, 6]
z = [7, 8, 9]
np.concatenate([x, y, z])

array([1, 2, 3, 4, 5, 6, 7, 8, 9])

첫 번째 인수는 연결할 배열의 목록 또는 튜플입니다.
또한 다차원 배열의 경우 결과가 연결될 축을 지정할 수 있는 `axis` 키워드를 사용합니다.

In [5]:
x = [[1, 2],
     [3, 4]]
np.concatenate([x, x], axis=1)

array([[1, 2, 1, 2],
       [3, 4, 3, 4]])

## pd.concat을 사용한 간단한 연결

`pd.concat` 함수는 `np.concatenate`와 유사한 구문을 제공하지만 잠시 논의할 여러 옵션을 포함합니다.

``파이썬
# Pandas v1.3.5의 서명
pd.concat(objs, 축=0, 조인='외부',ignore_index=False, 키=없음,
레벨=없음, 이름=없음, verify_integrity=False,
정렬=False, 복사=True)
````

`pd.concat`은 `np.concatenate`를 배열의 간단한 연결에 사용할 수 있는 것처럼 `Series` 또는 `DataFrame` 객체의 간단한 연결에 사용할 수 있습니다.

In [6]:
ser1 = pd.Series(['A', 'B', 'C'], index=[1, 2, 3])
ser2 = pd.Series(['D', 'E', 'F'], index=[4, 5, 6])
pd.concat([ser1, ser2])

1    A
2    B
3    C
4    D
5    E
6    F
dtype: object

또한 ``DataFrame``과 같은 고차원 개체를 연결하는 데에도 작동합니다.

In [7]:
df1 = make_df('AB', [1, 2])
df2 = make_df('AB', [3, 4])
display('df1', 'df2', 'pd.concat([df1, df2])')

,A,B
1,A1,B1
2,A2,B2
,A,B
3,A3,B3
4,A4,B4
,A,B
1,A1,B1
2,A2,B2
3,A3,B3
4,A4,B4


기본 동작은 `DataFrame`(예: `axis=0`) 내에서 행 단위로 연결하는 것입니다.
`np.concatenate`와 마찬가지로 `pd.concat`은 연결이 수행될 축을 지정할 수 있습니다.
다음 예를 고려하십시오.

In [8]:
df3 = make_df('AB', [0, 1])
df4 = make_df('CD', [0, 1])
display('df3', 'df4', "pd.concat([df3, df4], axis='columns')")

df3
    A   B
0  A0  B0
1  A1  B1

df4
    C   D
0  C0  D0
1  C1  D1

pd.concat([df3, df4], axis='columns')
    A   B   C   D
0  A0  B0  C0  D0
1  A1  B1  C1  D1

``axis=1``을 동일하게 지정할 수도 있습니다. 여기서는 보다 직관적인 ``axis='columns'``를 사용했습니다.

### 중복 인덱스

`np.concatenate`와 `pd.concat`의 중요한 차이점 중 하나는 Pandas 연결이 결과에 중복된 색인이 있더라도 *인덱스를 보존*한다는 것입니다!
다음의 짧은 예를 고려해보세요:

In [9]:
x = make_df('AB', [0, 1])
y = make_df('AB', [2, 3])
y.index = x.index  # make indices match
display('x', 'y', 'pd.concat([x, y])')

,A,B
0,A0,B0
1,A1,B1
,A,B
0,A2,B2
1,A3,B3
,A,B
0,A0,B0
1,A1,B1
0,A2,B2
1,A3,B3


결과에서 반복되는 인덱스를 확인하세요.
이는 ``DataFrame`` 내에서는 유효하지만 결과는 바람직하지 않은 경우가 많습니다.
`pd.concat`은 이를 처리하는 몇 가지 방법을 제공합니다.

#### 반복된 색인을 오류로 처리

`pd.concat` 결과의 인덱스가 겹치지 않는지 간단히 확인하려면 `verify_integrity` 플래그를 포함하면 됩니다.
이를 'True'로 설정하면 중복된 인덱스가 있는 경우 연결에서 예외가 발생합니다.
다음은 명확성을 위해 오류 메시지를 잡아서 인쇄하는 예입니다.

In [10]:
try:
    pd.concat([x, y], verify_integrity=True)
except ValueError as e:
    print("ValueError:", e)

ValueError: Indexes have overlapping values: Int64Index([0, 1], dtype='int64')


#### 색인 무시

때로는 인덱스 자체가 중요하지 않으므로 단순히 무시하는 것이 좋습니다.
이 옵션은 `ignore_index` 플래그를 사용하여 지정할 수 있습니다.
이를 `True`로 설정하면 연결로 인해 결과 `DataFrame`에 대한 새로운 정수 인덱스가 생성됩니다.

In [11]:
display('x', 'y', 'pd.concat([x, y], ignore_index=True)')

,A,B
0,A0,B0
1,A1,B1
,A,B
0,A2,B2
1,A3,B3
,A,B
0,A0,B0
1,A1,B1
2,A2,B2
3,A3,B3


#### MultiIndex 키 추가

또 다른 옵션은 `keys` 옵션을 사용하여 데이터 소스에 대한 레이블을 지정하는 것입니다. 결과는 데이터를 포함하는 계층적으로 색인화된 시리즈가 됩니다.

In [12]:
display('x', 'y', "pd.concat([x, y], keys=['x', 'y'])")

x
    A   B
0  A0  B0
1  A1  B1

y
    A   B
0  A2  B2
1  A3  B3

pd.concat([x, y], keys=['x', 'y'])
      A   B
x 0  A0  B0
  1  A1  B1
y 0  A2  B2
  1  A3  B3

[계층적 인덱싱](03.05-Hierarchical-Indexing.ipynb)에서 설명한 도구를 사용하여 이 곱셈 인덱스 `DataFrame`을 우리가 관심 있는 표현으로 변환할 수 있습니다.

### 조인을 사용한 연결

방금 살펴본 짧은 예에서는 주로 ``DataFrame``을 공유 열 이름과 연결했습니다.
실제로 서로 다른 소스의 데이터는 서로 다른 열 이름 세트를 가질 수 있으며 `pd.concat`은 이 경우 여러 옵션을 제공합니다.
일부(전부는 아님!) 열을 공통으로 갖는 다음 두 개의 ``DataFrame``을 연결하는 것을 고려해 보세요.

In [13]:
df5 = make_df('ABC', [1, 2])
df6 = make_df('BCD', [3, 4])
display('df5', 'df6', 'pd.concat([df5, df6])')

df5
    A   B   C
1  A1  B1  C1
2  A2  B2  C2

df6
    B   C   D
3  B3  C3  D3
4  B4  C4  D4

pd.concat([df5, df6])
     A   B   C    D
1   A1  B1  C1  NaN
2   A2  B2  C2  NaN
3  NaN  B3  C3   D3
4  NaN  B4  C4   D4

기본 동작은 사용 가능한 데이터가 없는 항목을 NA 값으로 채우는 것입니다.
이를 변경하려면 `concat` 함수의 `join` 매개변수를 조정하면 됩니다.
기본적으로 조인은 입력 열의 합집합(`join='outer'`)이지만 `join='inner'`를 사용하여 이를 열의 교차로 변경할 수 있습니다.

In [14]:
display('df5', 'df6',
        "pd.concat([df5, df6], join='inner')")

df5
    A   B   C
1  A1  B1  C1
2  A2  B2  C2

df6
    B   C   D
3  B3  C3  D3
4  B4  C4  D4

pd.concat([df5, df6], join='inner')
    B   C
1  B1  C1
2  B2  C2
3  B3  C3
4  B4  C4

또 다른 유용한 패턴은 삭제할 열을 더 세밀하게 제어하기 위해 연결하기 전에 'reindex' 메서드를 사용하는 것입니다.

In [15]:
pd.concat([df5, df6.reindex(df5.columns, axis=1)])

,A,B,C
1,A1,B1,C1
2,A2,B2,C2
3,NaN,B3,C3
4,NaN,B4,C4


### 추가 방법

직접 배열 연결이 매우 일반적이기 때문에 `Series` 및 `DataFrame` 객체에는 더 적은 키 입력으로 동일한 작업을 수행할 수 있는 `append` 메서드가 있습니다.
예를 들어 `pd.concat([df1, df2])` 대신 `df1.append(df2)`를 사용할 수 있습니다.

In [16]:
display('df1', 'df2', 'df1.append(df2)')

,A,B
1,A1,B1
2,A2,B2
,A,B
3,A3,B3
4,A4,B4
,A,B
1,A1,B1
2,A2,B2
3,A3,B3
4,A4,B4


파이썬(Python) 목록의 'append' 및 'extend' 메서드와 달리 Pandas의 'append' 메서드는 원본 개체를 수정하지 않는다는 점을 명심하세요. 대신 결합된 데이터로 새 개체를 만듭니다.
이는 또한 새로운 인덱스 *및* 데이터 버퍼 생성을 포함하기 때문에 매우 효율적인 방법이 아닙니다.
따라서 여러 개의 '추가' 작업을 수행하려는 경우 일반적으로 'DataFrame' 개체 목록을 작성하고 이를 모두 한 번에 'concat' 함수에 전달하는 것이 좋습니다.

다음 장에서는 여러 소스의 데이터를 결합하는 보다 강력한 접근 방식인 `pd.merge`에 구현된 데이터베이스 스타일 병합/조인을 살펴보겠습니다.
`concat`, `append` 및 관련 기능에 대한 자세한 내용은 Pandas 설명서의 ["병합, 조인, 연결 및 비교" 섹션](http://pandas.pydata.org/pandas-docs/stable/merging.html)을 참조하세요.